# 03 — DB Schema Validation & Connection Testing

**Purpose:** Verify the PostgreSQL database schema, test SQLAlchemy connections, and confirm all five ORM models (`experiments`, `runs`, `metrics`, `params`, `submissions`) behave as expected.

**What you'll learn:**
- How to connect to PostgreSQL via SQLAlchemy using the same `DATABASE_URL` as the FastAPI app
- How to inspect existing tables and confirm schema matches `db/models.py`
- How to insert and read back records for all five tables
- How to validate foreign key constraints and JSONB tag storage
- How to run the reference `init_schema.sql` programmatically

**Prerequisites:**
- PostgreSQL running (or use SQLite in-memory for quick smoke tests)
- `.env` file with `DATABASE_URL` set
- `sqlalchemy`, `psycopg2-binary` installed

---

## 1. Setup: Load Environment & Choose Database

The notebook supports two modes:
- **SQLite** (in-memory): quick smoke tests, no PostgreSQL needed
- **PostgreSQL**: full integration test using your `DATABASE_URL` from `.env`

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path('..') / 'backend' / '.env')

PG_URL = os.getenv('DATABASE_URL')

# Switch: set USE_SQLITE=True for a fully local, dependency-free test
USE_SQLITE = not bool(PG_URL)

if USE_SQLITE:
    DB_URL = "sqlite:///:memory:"
    print("Mode: SQLite in-memory (no PostgreSQL required)")
else:
    DB_URL = PG_URL
    print(f"Mode: PostgreSQL at {DB_URL.split('@')[-1] if '@' in DB_URL else DB_URL}")

## 2. Create SQLAlchemy Engine & Import Models

We import models directly from the backend source tree so this notebook always reflects the current schema.

In [ ]:
import sys
sys.path.insert(0, str(Path('..') / 'backend'))

from sqlalchemy import create_engine, inspect, text
from sqlalchemy.orm import sessionmaker

engine = create_engine(
    DB_URL,
    connect_args={"check_same_thread": False} if USE_SQLITE else {},
    echo=False,
)

Session = sessionmaker(bind=engine)

# Import ORM models (Base must know about all models before create_all)
from db.database import Base
from db import models  # noqa: F401  — registers all model classes with Base

# Create all tables
Base.metadata.create_all(bind=engine)
print("Tables created (or already exist)")

## 3. Inspect Schema: Confirm All 5 Tables Exist

In [ ]:
inspector = inspect(engine)
existing_tables = inspector.get_table_names()

EXPECTED_TABLES = ["experiments", "runs", "metrics", "params", "submissions"]

print("=== Table Existence Check ===")
all_ok = True
for table in EXPECTED_TABLES:
    found = table in existing_tables
    status = "OK" if found else "MISSING"
    if not found:
        all_ok = False
    print(f"  {status}  {table}")

print()
print("All tables present:" , all_ok)
assert all_ok, "Schema validation failed — missing tables!"

## 4. Inspect Columns for Each Table

Confirm column names, types, and nullability match `db/models.py`.

In [ ]:
import pandas as pd

for table in EXPECTED_TABLES:
    columns = inspector.get_columns(table)
    df = pd.DataFrame(columns)[['name', 'type', 'nullable', 'default']]
    print(f"\n--- {table.upper()} ---")
    print(df.to_string(index=False))

## 5. CRUD Test — Insert Sample Records

Insert one record per table and confirm round-trip integrity.

In [ ]:
from datetime import datetime, timezone
from db.models import Experiment, Run, Metric, Param, Submission

NOW = datetime.now(timezone.utc)

with Session() as session:
    # ---- Experiment ----
    exp = Experiment(
        experiment_id="nb_test_001",
        name="notebook_validation_experiment",
        artifact_location="dbfs:/mlflow/experiments/nb_test_001",
        lifecycle_stage="active",
        tags={"team": "notebook_test", "brand": "TestDrug"},
        created_at=NOW,
        synced_at=NOW,
    )
    session.merge(exp)  # merge = upsert so re-running doesn't fail

    # ---- Run ----
    run = Run(
        run_id="nb_run_001",
        experiment_id="nb_test_001",
        status="FINISHED",
        start_time=NOW,
        end_time=NOW,
        created_at=NOW,
    )
    session.merge(run)
    session.commit()

    # ---- Metric ----
    metric = Metric(run_id="nb_run_001", key="accuracy", value=0.95, timestamp=int(NOW.timestamp() * 1000))
    session.add(metric)

    # ---- Param ----
    param = Param(run_id="nb_run_001", key="n_estimators", value="250")
    session.add(param)

    # ---- Submission ----
    sub = Submission(
        run_id="nb_run_001",
        experiment_id="nb_test_001",
        submitted_by="notebook_tester@astrazeneca.com",
        status="PENDING",
        notes="Inserted from notebook 03_schema_validation",
        created_at=NOW,
        updated_at=NOW,
    )
    session.add(sub)
    session.commit()

    print("All 5 records inserted successfully")

## 6. Query & Validate Each Table

In [ ]:
with Session() as session:
    # Experiments
    exp_row = session.query(Experiment).filter_by(experiment_id="nb_test_001").first()
    assert exp_row is not None, "Experiment not found!"
    assert exp_row.name == "notebook_validation_experiment"
    assert exp_row.tags["team"] == "notebook_test"
    print(f"Experiment: {exp_row.name} | tags: {exp_row.tags}")

    # Runs
    run_row = session.query(Run).filter_by(run_id="nb_run_001").first()
    assert run_row is not None, "Run not found!"
    assert run_row.status == "FINISHED"
    assert run_row.experiment_id == "nb_test_001"
    print(f"Run: {run_row.run_id} | status: {run_row.status} | experiment: {run_row.experiment_id}")

    # Metrics
    metric_row = session.query(Metric).filter_by(run_id="nb_run_001", key="accuracy").first()
    assert metric_row is not None, "Metric not found!"
    assert abs(metric_row.value - 0.95) < 1e-9
    print(f"Metric: {metric_row.key} = {metric_row.value}")

    # Params
    param_row = session.query(Param).filter_by(run_id="nb_run_001", key="n_estimators").first()
    assert param_row is not None, "Param not found!"
    assert param_row.value == "250"
    print(f"Param: {param_row.key} = {param_row.value}")

    # Submissions
    sub_row = session.query(Submission).filter_by(run_id="nb_run_001").first()
    assert sub_row is not None, "Submission not found!"
    assert sub_row.status == "PENDING"
    print(f"Submission: id={sub_row.submission_id} | status={sub_row.status} | by={sub_row.submitted_by}")

print("\nAll assertions passed — schema is valid!")

## 7. Foreign Key Constraint Test

Inserting a run with a non-existent `experiment_id` should raise an error (PostgreSQL enforces FK constraints; SQLite requires `PRAGMA foreign_keys=ON`).

In [ ]:
from sqlalchemy.exc import IntegrityError

if not USE_SQLITE:
    try:
        with Session() as session:
            bad_run = Run(
                run_id="bad_run_fk_test",
                experiment_id="DOES_NOT_EXIST",
                status="RUNNING",
                created_at=NOW,
            )
            session.add(bad_run)
            session.commit()
        print("FK constraint NOT enforced (unexpected)")
    except IntegrityError as e:
        print(f"FK constraint correctly enforced: {type(e).__name__}")
else:
    print("SQLite mode: skipping FK constraint test (requires PostgreSQL)")

## 8. Row Counts Summary

In [ ]:
with Session() as session:
    counts = {
        "experiments": session.query(Experiment).count(),
        "runs":        session.query(Run).count(),
        "metrics":     session.query(Metric).count(),
        "params":      session.query(Param).count(),
        "submissions": session.query(Submission).count(),
    }

print("=== Row Counts ===")
for table, count in counts.items():
    print(f"  {table:<15} {count:>5} row(s)")

## 9. Cleanup (Optional)

Remove test records inserted in this notebook. Uncomment if you want a clean database after running.

In [ ]:
# UNCOMMENT to clean up test data

# with Session() as session:
#     session.query(Submission).filter_by(run_id="nb_run_001").delete()
#     session.query(Metric).filter_by(run_id="nb_run_001").delete()
#     session.query(Param).filter_by(run_id="nb_run_001").delete()
#     session.query(Run).filter_by(run_id="nb_run_001").delete()
#     session.query(Experiment).filter_by(experiment_id="nb_test_001").delete()
#     session.commit()
#     print("Test records cleaned up")

## 10. Summary

This notebook validated:
- All 5 tables exist in the database with correct columns
- CRUD operations work for `experiments`, `runs`, `metrics`, `params`, and `submissions`
- JSONB tag storage and retrieval works correctly
- Foreign key constraints are enforced (PostgreSQL)

The schema in `db/models.py` and `db/migrations/init_schema.sql` are consistent.

**You're ready to run the FastAPI backend!** Follow the `README.md` at the project root for full setup instructions.